In [ ]:
import scipy as sp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import skimage as ski

In [ ]:
mat = sp.io.loadmat("datasets/seville/world5000_gray.mat")
X, Y, Z = mat['X'], mat['Y'], mat['Z'] #units of decimeter
colp = mat['colp']
green = np.clip(255-np.mean(colp, axis=1) * 255, 0, 255).astype(np.uint8)
Z[Z < 0] = -Z[Z < 0]  # make all heights non-negative
base_length = np.sqrt(np.diff(X[:,:2], axis=1)**2 + np.diff(Y[:,:2], axis=1)**2)
orientation = np.arctan2(np.diff(Y[:,:2], axis=1), np.diff(X[:,:2], axis=1))*180/np.pi
elevation = np.arctan2(Z[:,2], ((X[:,2]-np.mean(X[:,:2], axis=1))**2 + (Y[:,2]-np.mean(Y[:,:2], axis=1))**2)**0.5)*180/np.pi
N = len(X)

In [ ]:
fig = plt.figure(figsize=(9, 4))
fig.suptitle(f'Seville 3D reconstruction (n={N} objects)', fontsize=10)

# Create a 2x3 grid
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.8], height_ratios=[1, 1], wspace=0.2, hspace=0.4)

# Scatter 1: height
ax1 = fig.add_subplot(gs[:, 0])  # spans both rows
sc1 = ax1.scatter(np.mean(X, axis=1), np.mean(Y, axis=1), c=np.mean(Z, axis=1),
                  cmap='plasma', s=5)
ax1.set_xlabel('X [dm]')
ax1.set_ylabel('Y [dm]')
ax1.set_aspect('equal')
cbar1 = plt.colorbar(sc1, ax=ax1, label='Height [dm]', orientation='horizontal', pad=0.2, shrink=0.6)

# Scatter 2: green
ax2 = fig.add_subplot(gs[:, 1])  # spans both rows
sc2 = ax2.scatter(np.mean(X, axis=1), np.mean(Y, axis=1), c=green,
                  cmap='Greens', s=5, vmin=0, vmax=255)
ax2.set_xlabel('X [dm]')
ax2.set_aspect('equal')
cbar2 = plt.colorbar(sc2, ax=ax2, label='Green [0-255]', orientation='horizontal', pad=0.2, shrink=0.6)

# Histogram 1: orientation (top-right)
ax3 = fig.add_subplot(gs[0, 2])
orientation_hist,orientation_bins,_ = ax3.hist(orientation, bins=50, color='gray', density=True)
ax3.set_title('probability density', fontsize = 10)

# Histogram 2: elevation (bottom-right)
ax4 = fig.add_subplot(gs[1, 2])
elevation_hist,elevation_bins,_ = ax4.hist(elevation, bins=50, color='pink', density=True)

ax3.set_xlabel('Orientation [deg]', fontsize = 10)
ax4.set_xlabel('Elevation [deg]', fontsize = 10)

sns.despine(ax = ax3, left = True, right = False)
sns.despine(ax = ax4, left = True, right = False)
plt.savefig('plots/seville_statistics.png', dpi=300)

# Generate grids

In [ ]:
grid_size = (50, 50)  # x bins, y bins
from scipy.stats import binned_statistic_2d

pixel_area = (np.max(np.mean(X[:,:2], axis=1))-np.min(np.mean(X[:,:2], axis=1)))/grid_size[0] * (np.max(np.mean(Y[:,:2], axis=1))-np.min(np.mean(Y[:,:2], axis=1)))/grid_size[1]  # dm^2

# Compute histogram
H, Xvals, Yvals = np.histogram2d(np.mean(X, axis=1), np.mean(Y, axis=1), bins=grid_size)

#compute density in objects/dm^2
Density = (H / pixel_area).T
print(f'mean density: {np.mean(Density):.2f} objects/dm^2, max density: {np.max(Density):.2f} objects/dm^2')

Height, _, _, _ = binned_statistic_2d(np.mean(X, axis=1), np.mean(Y, axis=1), np.mean(Z, axis=1), statistic='mean', bins=grid_size)
Height = (np.nan_to_num(Height)).T  # Replace NaNs with zero for plotting

Green, _, _, _ = binned_statistic_2d(np.mean(X, axis=1), np.mean(Y, axis=1), green, statistic='mean', bins=grid_size)
Green = (np.nan_to_num(Green)).T  # Replace NaNs with zero for plotting

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (9,4), gridspec_kw={'width_ratios': [1, 1, 1]}, )
cm = ax[0].imshow(Density, origin='lower', cmap='viridis', extent=[Xvals[0], Xvals[-1], Yvals[0], Yvals[-1]])
plt.colorbar(cm, label='Density [objects/dm²]', shrink = 0.5, orientation='horizontal', pad=0.2)
ax[0].set_xlabel('X [dm]')
ax[0].set_ylabel('Y [dm]')
ax[0].set_aspect('equal')

cm = ax[1].imshow(Height, origin='lower', cmap='viridis', extent=[Xvals[0], Xvals[-1], Yvals[0], Yvals[-1]])
plt.colorbar(cm, label='Height [dm]', shrink = 0.5, orientation='horizontal', pad=0.2)
ax[1].set_xlabel('X [dm]')
ax[1].set_aspect('equal')

cm = ax[2].imshow(Green, origin='lower', cmap='Greens', extent=[Xvals[0], Xvals[-1], Yvals[0], Yvals[-1]])
plt.colorbar(cm, label='Intensity [0-255]', shrink = 0.5, orientation='horizontal', pad=0.2)
ax[2].set_xlabel('X [dm]')
ax[2].set_aspect('equal')
plt.savefig('plots/heatmaps.png', dpi=300)

# Save the data

In [ ]:
# Save the grids
import pandas as pd
DensityDf = pd.DataFrame(Density, columns=np.round(Xvals[1:],2), index=np.round(Yvals[1:],2))
HeightDf = pd.DataFrame(Height, columns=np.round(Xvals[1:],2), index=np.round(Yvals[1:],2))
GreenDf = pd.DataFrame(Green, columns=np.round(Xvals[1:],2), index=np.round(Yvals[1:],2))
OrientationDf = pd.DataFrame(orientation_hist, columns=['probability_density'], index=np.round(orientation_bins[1:],2))
ElevationDf = pd.DataFrame(elevation_hist, columns=['probability_density'], index=np.round(elevation_bins[1:],2))

DensityDf.to_csv('datasets/seville/statistics/Density_objects_dm2.csv')
HeightDf.to_csv('datasets/seville/statistics/Height_dm.csv')
GreenDf.to_csv('datasets/seville/statistics/Intensity_0-255.csv')
OrientationDf.to_csv('datasets/seville/statistics/Orientation_probability_density.csv')
ElevationDf.to_csv('datasets/seville/statistics/Elevation_probability_density.csv')